# Event Impact — DWH exports (run in Colab)

BigQuery is refused at Google's edge from the office network, but works from Colab (as `BQ_for_trading.ipynb` shows).
This notebook produces the two files the Event Impact tab reads automatically:

1. `rebate_payout_export.csv` — per-trade `rebate_payout` with the login (feeds *Rebates lifetime / month*, *Net revenue*).
2. `account_links.csv` — `trading_account_number` → `primary_trading_account_number` (feeds *Primary account / Sub-accounts*).

Drop both into `notebook/docs/` (or Downloads). The next **Analyze** picks them up — no restart.

In [ ]:
from google.cloud import bigquery
from google.colab import auth, files
import pandas as pd, re

auth.authenticate_user()
PROJECT = 'zfx-dwh-prod'
client = bigquery.Client(project=PROJECT)
print('authenticated to', PROJECT)

In [ ]:
# 1. WHERE do the fields live? Search every dataset's INFORMATION_SCHEMA.
PATTERN = r'rebate_payout|primary_trading_account|trading_account_number|rebate'
hits = []
for ds in client.list_datasets():
    d = ds.dataset_id
    try:
        sql = f"""
        SELECT table_name, column_name, data_type
        FROM `{PROJECT}.{d}.INFORMATION_SCHEMA.COLUMNS`
        WHERE REGEXP_CONTAINS(LOWER(column_name), r'{PATTERN}')
        """
        for r in client.query(sql).result():
            hits.append((d, r.table_name, r.column_name, r.data_type))
    except Exception as e:
        print(f'[{d}] skipped: {str(e)[:80]}')
cols = pd.DataFrame(hits, columns=['dataset', 'table', 'column', 'type'])
pd.set_option('display.max_rows', 300)
cols

In [ ]:
# 2. Pick the source tables (edit the two names below if the search found several).
reb_tables = cols[cols['column'].str.lower() == 'rebate_payout'][['dataset', 'table']].drop_duplicates()
lnk_tables = cols[cols['column'].str.lower() == 'primary_trading_account_number'][['dataset', 'table']].drop_duplicates()
print('tables with rebate_payout:'); print(reb_tables.to_string(index=False))
print('\ntables with primary_trading_account_number:'); print(lnk_tables.to_string(index=False))

REBATE_TABLE = f"{PROJECT}.{reb_tables.iloc[0]['dataset']}.{reb_tables.iloc[0]['table']}" if len(reb_tables) else None
LINK_TABLE = f"{PROJECT}.{lnk_tables.iloc[0]['dataset']}.{lnk_tables.iloc[0]['table']}" if len(lnk_tables) else None
print('\nREBATE_TABLE =', REBATE_TABLE); print('LINK_TABLE   =', LINK_TABLE)

def columns_of(table):
    return [f.name for f in client.get_table(table).schema]
if REBATE_TABLE: print('\nrebate table columns:', columns_of(REBATE_TABLE))
if LINK_TABLE: print('\nlink table columns:', columns_of(LINK_TABLE))

In [ ]:
# 3. REBATES: one row per trade, last 730 days (the app's retention). The login column is
#    detected; the time column too (close time preferred). Adjust if the print above shows other names.
assert REBATE_TABLE, 'no table with rebate_payout found'
rc = columns_of(REBATE_TABLE)
login_col = next(c for c in rc if c.lower() in ('login', 'trading_account_number', 'account_number', 'mt_login', 'account'))
time_col = next((c for c in rc if re.search(r'close', c, re.I)), next((c for c in rc if re.search(r'time|date', c, re.I)), None))
server_col = next((c for c in rc if re.search(r'server|platform|database', c, re.I)), None)
extra = ([server_col] if server_col else []) + ([time_col] if time_col else [])
where = f"WHERE {time_col} >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 730 DAY)" if time_col else ''
sql = f"""
SELECT {login_col} AS login, {', '.join(extra + ['rebate_payout'])}
FROM `{REBATE_TABLE}`
{where}
  AND rebate_payout IS NOT NULL AND rebate_payout <> 0
""" if where else f"SELECT {login_col} AS login, {', '.join(extra + ['rebate_payout'])} FROM `{REBATE_TABLE}` WHERE rebate_payout IS NOT NULL AND rebate_payout <> 0"
print(sql)
reb = client.query(sql).to_dataframe()
print(f'{len(reb):,} rebate rows, {reb.login.nunique():,} logins, total {reb.rebate_payout.sum():,.2f}')
reb.to_csv('rebate_payout_export.csv', index=False)
files.download('rebate_payout_export.csv')

In [ ]:
# 4. ACCOUNT LINKS: every trading account with its primary.
assert LINK_TABLE, 'no table with primary_trading_account_number found'
lc = columns_of(LINK_TABLE)
acct_col = next(c for c in lc if c.lower() in ('trading_account_number', 'login', 'account_number', 'mt_login'))
server_col = next((c for c in lc if re.search(r'server|platform|database', c, re.I)), None)
sel = [f'{acct_col} AS trading_account_number', 'primary_trading_account_number'] + ([f'{server_col} AS server'] if server_col else [])
sql = f"""
SELECT DISTINCT {', '.join(sel)}
FROM `{LINK_TABLE}`
WHERE primary_trading_account_number IS NOT NULL AND {acct_col} IS NOT NULL
"""
print(sql)
links = client.query(sql).to_dataframe()
multi = links.groupby('primary_trading_account_number').size()
print(f'{len(links):,} accounts, {links.primary_trading_account_number.nunique():,} primaries, {(multi > 1).sum():,} primaries with sub-accounts')
links.to_csv('account_links.csv', index=False)
files.download('account_links.csv')

Then: copy `rebate_payout_export.csv` and `account_links.csv` into `notebook/docs/` and press **Analyze** (or a NFP button) on the Event Impact tab. The data notes at the bottom of the tab confirm which source was used (`Rebate source` column, `Link source` column).